# 04 - Regularización en Regresión Logística y Generalización Multiclase

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno profundizaremos en dos aspectos avanzados fundamentales de la **Regresion Logistica**: la introduccion de terminos de **regularizacion ($L_1$ y $L_2$)** para controlar el sobreajuste y realizar seleccion automatica de caracteristicas mediante el hiperparametro $C$, y la extension matematica a problemas de **clasificacion multiclase** ($K > 2$) mediante las estrategias One-vs-Rest (OvR) y la formulacion **Multinomial / Softmax**, utilizando los datasets estandar de Scikit-Learn `load_breast_cancer` y `load_wine`.

---

### Contenidos
1. Regularizacion en Clasificacion: El Hiperparametro $C$
2. Penalizaciones $L_1$ vs. $L_2$ en Regresion Logistica
3. Seleccion Embebida de Caracteristicas con Norma $L_1$ (`solver='saga'`)
4. Optimizacion de $C$ mediante `LogisticRegressionCV`
5. Generalizacion a Problemas Multiclase ($K > 2$)
6. Enfoque One-vs-Rest (OvR) frente a Regresion Multinomial (Softmax)
7. Caso de Estudio Multiclase: Dataset de Vinos (`load_wine`)
8. Matriz de Confusion Multiclase y Metricas Macro / Weighted
9. Conclusiones y Resumen del Modulo


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Regularización en Clasificación: El Hiperparametro $C$

En Scikit-Learn, `LogisticRegression` aplica regularizacion por defecto. A diferencia de Ridge o Lasso (donde se utiliza $\alpha$ como multiplicador directo de la penalizacion), en regresion logistica se define el hiperparametro **$C$**, que representa la **fuerza inversa de la regularizacion**:

$$C = \frac{1}{\lambda}$$

La funcion objetivo minimizada tiene la forma:

$$\min_\beta C \cdot J_{\text{Log-Loss}}(\beta) + R(\beta)$$

Donde $R(\beta)$ es el termino de penalizacion:
- **Penalizacion $L_2$**: $R(\beta) = \frac{1}{2} \|\beta\|_2^2 = \frac{1}{2} \sum_{j=1}^p \beta_j^2$
- **Penalizacion $L_1$**: $R(\beta) = \|\beta\|_1 = \sum_{j=1}^p |\beta_j|$

### Interpretacion de $C$:
- **$C$ muy pequeno** (ej. $C = 0.001$): Penalizacion muy intensa $\to$ Coeficientes fuertemente comprimidos hacia cero $\to$ Modelo muy simple, menor varianza, previene sobreajuste.
- **$C$ muy grande** (ej. $C = 1000$): Penalizacion insignificante $\to$ Coeficientes libres $\to$ Se aproxima al estimador clasico de Maxima Verosimilitud sin regularizar.


## 2. Selección Embebida de Caracteristicas con Norma $L_1$

Al aplicar una penalizacion $L_1$ en regresion logistica (utilizando el optimizador `solver='saga'` o `solver='liblinear'`), el modelo actua como selector de variables, anulando exactamente los coeficientes superfluos o redundantes.


In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_bc = cancer.frame.drop(columns=['target'])
y_bc = cancer.frame['target']

X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.20, random_state=42, stratify=y_bc
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bc)
X_test_sc = scaler.transform(X_test_bc)

# Evaluacion de coeficientes para una malla de C con penalizacion L1
valores_C = np.logspace(-2, 2, 20)
conteo_variables_retenidas = []
coeficientes_l1 = []

for c in valores_C:
    clf_l1 = LogisticRegression(penalty='l1', C=c, solver='liblinear', random_state=42)
    clf_l1.fit(X_train_sc, y_train_bc)
    coefs = clf_l1.coef_[0]
    coeficientes_l1.append(coefs)
    conteo_variables_retenidas.append(np.sum(coefs != 0))

coeficientes_l1 = np.array(coeficientes_l1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grafico 1: Trayectoria de coeficientes segun C
for j in range(X_bc.shape[1]):
    axes[0].plot(valores_C, coeficientes_l1[:, j], linewidth=1.3, alpha=0.8)
axes[0].set_xscale('log')
axes[0].set_xlabel('Parametro C (Escala Logaritmica)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Magnitud de Coeficientes', fontsize=11, fontweight='bold')
axes[0].set_title('Camino de Regularizacion L1 en Regresion Logistica', fontsize=12, fontweight='bold')
axes[0].axhline(0, color='black', linestyle='--', alpha=0.5)

# Grafico 2: Numero de variables activas vs C
axes[1].plot(valores_C, conteo_variables_retenidas, marker='o', color='#2b5c8f', linewidth=2)
axes[1].set_xscale('log')
axes[1].set_xlabel('Parametro C (Escala Logaritmica)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Numero de Variables con Coeficiente != 0', fontsize=11, fontweight='bold')
axes[1].set_title('Numero de Variables Retenidas segun Fuerza C', fontsize=12, fontweight='bold')
axes[1].axhline(X_bc.shape[1], color='gray', linestyle=':', label='Total (30 variables)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Optimización Automática con `LogisticRegressionCV`

Scikit-Learn provee `LogisticRegressionCV` para encontrar el valor optimo de $C$ mediante validacion cruzada estratificada k-fold.


In [ ]:
# Busqueda de C optimo con validacion cruzada (cv=5)
pipeline_log_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('log_cv', LogisticRegressionCV(Cs=np.logspace(-3, 3, 50), cv=5, scoring='accuracy',
                                    l1_ratios=(0.0,), max_iter=1000, random_state=42))
])

pipeline_log_cv.fit(X_train_bc, y_train_bc)

c_optimo = pipeline_log_cv.named_steps['log_cv'].C_[0]
acc_opt = pipeline_log_cv.score(X_test_bc, y_test_bc)

print(f"Mejor parametro C seleccionado por validacion cruzada: {c_optimo:.4f}")
print(f"Exactitud (Accuracy) obtenida en conjunto de prueba:    {acc_opt*100:.2f}%")


## 4. Generalización a Clasificación Multiclase ($K > 2$)

Cuando la variable objetivo puede tomar una de $K > 2$ categorias posibles (por ejemplo, tres especies botánicas o tres tipos de vino), la regresion logistica binaria se generaliza mediante dos estrategias principales:

### 1. One-vs-Rest (OvR / One-vs-All)
- Entrena $K$ clasificadores binarios independientes.
- El clasificador $k$ aprende a distinguir la clase $k$ frente a todas las demas combinadas.
- En la prediccion, se evaluan los $K$ modelos y se asigna la clase con mayor puntuacion de probabilidad:
  $$\hat{y} = \arg\max_{k \in \{1, \dots, K\}} P_k(Y = k \mid X)$$

### 2. Regresión Multinomial (Softmax)
Modela simultaneamente la distribucion de probabilidad conjunta sobre las $K$ clases mediante la funcion **Softmax**:

$$P(Y = k \mid X = x) = \frac{e^{\beta_k^T x}}{\sum_{j=1}^K e^{\beta_j^T x}}$$

- La suma de las probabilidades para las $K$ clases suma estrictamente 1:
  $$\sum_{k=1}^K P(Y = k \mid X = x) = 1$$
- La funcion de perdida es la **Entropia Cruzada Multiclase (Categorical Cross-Entropy)**:
  $$J(\beta) = -\frac{1}{n} \sum_{i=1}^n \sum_{k=1}^K \mathbb{I}(y_i = k) \ln(\hat{p}_{ik})$$


## 5. Caso de Estudio Multiclase: Dataset de Vinos (`load_wine`)

Utilizaremos el dataset de vinos de Scikit-Learn, el cual contiene 178 muestras con 13 caracteristicas quimicas (alcohol, acido malico, ceniza, alcalinidad, magnesio, fenoles totales, flavonoides, intensidad de color, etc.) derivadas de 3 cultivares italianos distintos:
- Clase 0: Cultivar 1 (59 muestras)
- Clase 1: Cultivar 2 (71 muestras)
- Clase 2: Cultivar 3 (48 muestras)


In [ ]:
wine = load_wine(as_frame=True)
df_wine = wine.frame

print(f"Dimensiones del dataset de vinos: {df_wine.shape}")
print(f"Clases objetivo: {list(wine.target_names)}")
print("\nDistribucion de muestras por clase:")
print(df_wine['target'].value_counts().sort_index())


## 6. Ajuste de Regresión Logística Softmax (Multinomial)

Construimos un `Pipeline` con estandarizacion y ajustamos un modelo multiclase `multi_class='multinomial'` (o predeterminado en `lbfgs`).


In [ ]:
X_w = df_wine.drop(columns=['target'])
y_w = df_wine['target']

X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(
    X_w, y_w, test_size=0.25, random_state=42, stratify=y_w
)

pipeline_wine = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42))
])

pipeline_wine.fit(X_tr_w, y_tr_w)

y_pred_w = pipeline_wine.predict(X_te_w)
acc_w = accuracy_score(y_te_w, y_pred_w)

print(f"Exactitud en clasificacion multiclase (Conjunto de prueba): {acc_w*100:.2f}%")


In [ ]:
cm_w = confusion_matrix(y_te_w, y_pred_w)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_w, display_labels=wine.target_names)
disp.plot(ax=ax, cmap='Greens', values_format='d')
plt.title('Matriz de Confusion Multiclase (Softmax)', fontsize=12, fontweight='bold')
plt.grid(False)
plt.show()

print("Reporte de Clasificacion Multiclase:")
print(classification_report(y_te_w, y_pred_w, target_names=wine.target_names, digits=4))


## 7. Conclusiones y Resumen del Módulo

### Hallazgos Principales:
1. **El Hiperparametro $C$**: Controla el equilibrio sesgo-varianza en clasificacion de manera inversa a $\alpha$. Valores reducidos de $C$ robustecen el modelo contra sobreajuste.
2. **Seleccion de Variables con $L_1$**: Permite reducir modelos con decenas de variables clinicas o fisicas a subconjuntos compactos y explicables.
3. **Generalizacion Multiclase**: La funcion Softmax generaliza rigurosamente la sigmoide, permitiendo calcular distribuciones probabilisticas coherentes sobre cualquier numero finito de categorias.
4. **Pipelines**: Mantener `StandardScaler` integrado antes de la clasificacion logistica garantiza convergencia numerica acelerada y comparabilidad valida de coeficientes.
